In [0]:
-- SQL cell
CREATE CATALOG IF NOT EXISTS demo;
CREATE SCHEMA IF NOT EXISTS demo.raw;
CREATE SCHEMA IF NOT EXISTS demo.curated;
CREATE SCHEMA IF NOT EXISTS demo.serving;
-- Create Bronze table with schema enforcement + CDF
CREATE TABLE IF NOT EXISTS demo.raw.orders_bronze (
  order_id       STRING,
  customer_id    STRING,
  order_ts       TIMESTAMP,
  amount         DOUBLE,
  status         STRING
)
USING DELTA
TBLPROPERTIES (
  delta.enableChangeDataFeed = true  -- CDF
);

-- First batch insert
INSERT INTO demo.raw.orders_bronze VALUES
('o1','c1','2026-07-20T10:00:00',100.0,'NEW'),
('o2','c2','2026-07-20T11:00:00',200.0,'NEW');

-- Second batch with an update and new row
INSERT INTO demo.raw.orders_bronze VALUES
('o1','c1','2026-07-20T12:00:00',120.0,'UPDATED'),
('o3','c3','2026-07-20T12:30:00',150.0,'NEW');

-- Inspect history (shows ACID commits / versions)
DESCRIBE HISTORY demo.raw.orders_bronze;

--Time Travel
-- As-of older version (replace N with version from DESCRIBE HISTORY)
SELECT * FROM demo.raw.orders_bronze VERSION AS OF 1;

-- Current snapshot
SELECT * FROM demo.raw.orders_bronze;

CREATE TABLE IF NOT EXISTS demo.curated.orders_silver (
  order_id       STRING,
  customer_id    STRING,
  order_date     DATE,
  order_ts       TIMESTAMP,
  amount         DOUBLE,
  status         STRING,
  load_ts        TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

--bronze-->silver
MERGE INTO demo.curated.orders_silver AS tgt
USING (
  SELECT
    order_id,
    customer_id,
    DATE(order_ts) AS order_date,
    order_ts,
    amount,
    status,
    current_timestamp() AS load_ts
  FROM demo.raw.orders_bronze
) AS src
ON tgt.order_id = src.order_id
WHEN MATCHED THEN UPDATE SET
  tgt.customer_id = src.customer_id,
  tgt.order_date  = src.order_date,
  tgt.order_ts    = src.order_ts,
  tgt.amount      = src.amount,
  tgt.status      = src.status,
  tgt.load_ts     = src.load_ts
WHEN NOT MATCHED THEN INSERT *
;
-- Try writing a row with an extra unexpected column (or wrong type)
INSERT INTO demo.curated.orders_silver (order_id, customer_id, order_date, order_ts, amount, status)
VALUES ('o_bad','c_bad', '2026-07-21', '2026-07-21T10:00:00', 'NOT_A_DOUBLE', 'NEW');
-- Expect a schema/type error instead of silent corruption

-- Evolve schema by adding new column
ALTER TABLE demo.curated.orders_silver ADD COLUMNS (source_system STRING);

-- Now write data with the new column
INSERT INTO demo.curated.orders_silver
(order_id, customer_id, order_date, order_ts, amount, status, load_ts, source_system)
VALUES ('o4','c4', '2026-07-21', '2026-07-21T09:00:00', 250.0, 'NEW', current_timestamp(), 'CRM');

CREATE TABLE IF NOT EXISTS demo.serving.orders_gold (
  order_date   DATE,
  customer_id  STRING,
  total_amount DOUBLE,
  order_count  BIGINT
)
USING DELTA
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);
--gold<-->silver
INSERT OVERWRITE TABLE demo.serving.orders_gold
SELECT
  order_date,
  customer_id,
  SUM(amount) AS total_amount,
  COUNT(*)    AS order_count
FROM demo.curated.orders_silver
GROUP BY order_date, customer_id;
--liquid clustering
-- 1) Create gold table with clustering from the start
CREATE OR REPLACE TABLE demo.serving.orders_gold (
  order_date   DATE,
  customer_id  STRING,
  total_amount DOUBLE,
  order_count  BIGINT
)
USING DELTA
CLUSTER BY (order_date, customer_id);
-- 2) Populate gold from silver
INSERT OVERWRITE TABLE demo.serving.orders_gold
SELECT
  order_date,
  customer_id,
  SUM(amount) AS total_amount,
  COUNT(*)    AS order_count
FROM demo.curated.orders_silver
GROUP BY order_date, customer_id;
-- 3) Compact + cluster
OPTIMIZE demo.serving.orders_gold;
--cdf
-- Read only changes from version X to latest for Silver table
SELECT *
FROM table_changes('demo.curated.orders_silver', X);
MERGE INTO demo.serving.orders_gold AS tgt
USING (
  SELECT
    order_date,
    customer_id,
    SUM(amount) AS total_amount,
    COUNT(*)    AS order_count
  FROM table_changes('demo.curated.orders_silver', X)
  WHERE _change_type IN ('insert','update_postimage')
  GROUP BY order_date, customer_id
) AS src
ON tgt.order_date = src.order_date AND tgt.customer_id = src.customer_id
WHEN MATCHED THEN UPDATE SET
  tgt.total_amount = src.total_amount,
  tgt.order_count  = src.order_count
WHEN NOT MATCHED THEN INSERT *;